# 3D Asset Optimization: Draco Compression

This notebook documents the process and results of optimizing 3D assets for the **Animatic AI** web application. To ensure fast loading times over network-constrained connections (e.g., Cloudflare Tunnel), we implemented **Google Draco** compression.

## Objectives
- Reduce the network payload of 3D models (.glb files).
- Document the batch processing pipeline using `gltf-pipeline`.
- Analyze the final size reduction and performance impact.

## 1. Context and Problem Statement

The initial set of 3D models used in the landing page (Sakura, Robot, Portfolio examples) had a total size of **252 MB**. 

**Challenges:**
- **Latency:** Over a 10 Mbps connection, loading 252 MB takes ~3.5 minutes.
- **UX:** Users experience significant lag or empty scenes while assets are fetching.
- **Memory:** Large uncompressed meshes consume more GPU memory on mobile devices.

## 2. Methodology: Draco Compression

We chose **Draco** because it provides high compression ratios for geometry (vertices, connectivity, normals) with minimal visual loss.

### Tooling
We used `gltf-pipeline`, a Node.js utility for processing glTF files.

```bash
# Installation
npm install -g gltf-pipeline
```

### Optimization Parameters
- **Draco compression:** Enabled (`-d`).
- **Compression level:** 10 (Maximum geometry compression).
- **Output format:** Binary GLB for single-file portability.

## 3. Batch Processing Script

The following Python script was used to automate the compression of all models in the `public/models/` directory.

In [ ]:
import os
import subprocess

def compress_models(input_dir, output_dir):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        
    for filename in os.listdir(input_dir):
        if filename.endswith(".glb"):
            input_path = os.path.join(input_dir, filename)
            output_path = os.path.join(output_dir, filename)
            
            print(f"Processing {filename}...")
            command = [
                "npx", "gltf-pipeline",
                "-i", input_path,
                "-o", output_path,
                "-d", # Enable Draco
                "--draco.compressionLevel", "10"
            ]
            
            subprocess.run(command, shell=True)

# Example usage (commented out to prevent accidental execution)
# compress_models("path/to/original", "path/to/optimized")

## 4. Results and Analysis

| Asset Group | Original Size | Optimized Size (Draco) | Reduction |
|-------------|---------------|------------------------|-----------|
| All Models  | 252.0 MB      | 57.5 MB                | **77.2%** |

### Visual Verification
Despite the 77% reduction, the visual quality of the textures and high-poly geometry (like the Sakura blossoms) remained intact due to Draco's intelligent quantization.

## 5. Web Integration

To support Draco-compressed models in the React frontend, we updated the `useGLTF` hook to include the Draco decoder path.

```typescript
// Example integration in React-Three-Fiber
const { scene } = useGLTF(
  "/models/sakura_optimized.glb", 
  "https://www.gstatic.com/draco/versioned/decoders/1.5.5/"
);
```

## Conclusion
Draco compression effectively solved the bottleneck for 3D asset delivery. The reduction to **57.5 MB** allows the application to load within seconds on standard connections, especially when combined with our global loading screen.